In [2]:
# Source - https://stackoverflow.com/a/77503061
# Posted by Rajnish Dubey
# Retrieved 2026-05-07, License - CC BY-SA 4.0

import os, gzip, shutil
dir_name = r'/Users/mmo/VSCode/Predicting-Tissue-Specific-UTR/data'
def gz_extract(directory):
    extension = ".gz"
    os.chdir(directory)
    for item in os.listdir(directory): # loop through items in dir
      if item.endswith(extension): # check for ".gz" extension
          gz_name = os.path.abspath(item) # get full path of files
          file_name = (os.path.basename(gz_name)).rsplit('.',1)[0] #get file name for file within
          with gzip.open(gz_name,"rb") as f_in, open(file_name,"wb") as f_out:
              shutil.copyfileobj(f_in, f_out)
          os.remove(gz_name) # delete zipped file      
gz_extract(dir_name)


In [1]:
import re
def extract_gtf_attribute(series, key):
    return series.str.extract(fr'{key} "([^"]+)"', expand=False)

def get_appris_rank(attr):
    match = re.search(r'tag "appris_principal_(\d)"', attr)
    if match: return int(match.group(1))
    if 'tag "appris_principal"' in attr: return 1
    return 10  # Lowest priority

def format_interval(row):
    return f"{row['seqname']}:{row['start']}-{row['end']}({row['strand']})"

In [ ]:
import pandas as pd
from pathlib import Path

gtf_path = Path('..') / 'data' / 'gencode.v49.primary_assembly.annotation.gtf'
rank_path = Path('..') / 'data' / 'gencode.v49.transcript_rankings.txt'

df = pd.read_csv(gtf_path, sep='\t', comment='#', header=None,
                 names=['seqname','source','feature','start','end','score','strand','frame','attribute'])

# Extract gene_id and transcript_id 
df['gene_id'] = extract_gtf_attribute(df['attribute'], 'gene_id')
df['transcript_id'] = extract_gtf_attribute(df['attribute'], 'transcript_id')

# Filter for ENST and Rank 1
ranks = pd.read_csv(rank_path, sep='\t', header=None, usecols=[1,3,4], names=['gene_id','rank','transcript_id'])
rank1_ensts = ranks.loc[(ranks['transcript_id'].str.startswith('ENST')) & (ranks['rank'] == 1), 'transcript_id'].unique()

# Select one transcript per gene based on APPRIS
transcripts = df[df['feature'] == 'transcript'].copy()
transcripts = transcripts[transcripts['transcript_id'].isin(rank1_ensts)].copy()
transcripts['appris_rank'] = transcripts['attribute'].apply(get_appris_rank)

final_best_ids = (
    transcripts.sort_values(['gene_id', 'appris_rank'])
    .drop_duplicates('gene_id')['transcript_id']
)

# Define start and stop codons
codons = df[
    (df['feature'].isin(['start_codon', 'stop_codon'])) & 
    (df['transcript_id'].isin(final_best_ids))
].copy()

codon_bounds = codons.groupby(['transcript_id', 'feature']).agg({'start': 'min', 'end': 'max'}).unstack()

# Classify UTR
feature_rows = df[
    (df['feature'].isin(['UTR', 'CDS'])) & 
    (df['transcript_id'].isin(final_best_ids))
].copy()

def classify_utr(row, codon_map):
    tid = row['transcript_id']
    if row['feature'] == 'CDS': return 'CDS'
    if tid not in codon_map.index: return 'UTR_Incomplete'
    
    try:
        start_c_start = codon_map.loc[tid, ('start', 'start_codon')]
        stop_c_end = codon_map.loc[tid, ('end', 'stop_codon')]
    except KeyError:
        return 'UTR_Unknown'

    if row['strand'] == '+':
        if row['end'] <= start_c_start: return "5' UTR"
        if row['start'] >= stop_c_end: return "3' UTR"
    else: 
        # Invert for - strand
        if row['start'] >= start_c_start: return "5' UTR"
        if row['end'] <= stop_c_end: return "3' UTR"
    return "UTR_Internal"

feature_rows['feature'] = feature_rows.apply(lambda r: classify_utr(r, codon_bounds), axis=1)
feature_rows['interval'] = feature_rows.apply(format_interval, axis=1)

# Format
final_features = ["5' UTR", "CDS", "3' UTR"]
initialDB_coords_df = (
    feature_rows[feature_rows['feature'].isin(final_features)]
    .groupby(['transcript_id', 'feature'], sort=False)['interval']
    .agg('; '.join) 
    .unstack(fill_value='')
    .reindex(columns=final_features)
    .reset_index()
)

initialDB_coords_df.head()

feature,transcript_id,5' UTR,CDS,3' UTR
0,ENST00000641515.2,chr1:65419-65433(+); chr1:65520-65564(+),chr1:65565-65573(+); chr1:69037-70005(+),
1,ENST00000426406.4,,chr1:450743-451678(-),chr1:450740-450742(-)
2,ENST00000332831.5,,chr1:685719-686654(-),chr1:685716-685718(-)
3,ENST00000616016.5,chr1:923923-924431(+),chr1:924432-924948(+); chr1:925922-926013(+); ...,
4,ENST00000327044.7,chr1:959241-959256(-),chr1:959215-959240(-); chr1:958929-959081(-); ...,chr1:944203-944696(-)


In [5]:
initialDB_coords_df.to_csv('../data/initialDB_coords.csv', index=False)

In [5]:
assembly_path = Path('..') / 'data' / 'GRCh38.primary_assembly.genome.fa'

def load_genome_fasta(path):
    genome = {}
    with open(path) as f:
        chrom = None
        seq_chunks = []
        for line in f:
            if line.startswith('>'):
                if chrom: genome[chrom] = ''.join(seq_chunks)
                chrom = line[1:].split()[0]
                seq_chunks = []
            else:
                seq_chunks.append(line.strip())
        if chrom: genome[chrom] = ''.join(seq_chunks)
    return genome

genome = load_genome_fasta(assembly_path)

In [ ]:
interval_pattern = re.compile(r'^(?P<chrom>[^:]+):(?P<start>\d+)-(?P<end>\d+)\((?P<strand>[+-])\)$')


def reverse_complement(seq):
    """Return the reverse complement of a DNA sequence."""
    return seq.translate(str.maketrans('ACGTacgtNn', 'TGCAtgcaNn'))[::-1]


def get_chrom_sequence(chrom):
    """Return the sequence for a chromosome, trying both 'chr' and non-'chr' versions."""
    if chrom in genome:
        return genome[chrom]
    alt_chrom = chrom[3:] if chrom.startswith('chr') else f'chr{chrom}'
    if alt_chrom in genome:
        return genome[alt_chrom]
    raise KeyError(f'Chromosome {chrom!r} not found in loaded genome')


def interval_to_sequence(interval):
    """Return the sequence for a given interval."""
    match = interval_pattern.match(interval)
    if match is None:
        raise ValueError(f'Could not parse interval {interval!r}')

    chrom = match.group('chrom')
    start = int(match.group('start'))
    end = int(match.group('end'))
    strand = match.group('strand')

    sequence = get_chrom_sequence(chrom)[start - 1:end]
    if strand == '-':
        sequence = reverse_complement(sequence)
    return sequence


def intervals_to_sequence(interval_string):
    """Return the sequences of intervals indicated by the coordinates. The coordinates are concatenated as the separation 
    before was only due to the fact they were on different exons.
    """
    if pd.isna(interval_string) or interval_string == '':
        return ''
    return ''.join(
        interval_to_sequence(interval.strip())
        for interval in interval_string.split(';')
        if interval.strip()
    )


sequence_cols = ["5' UTR", 'CDS', "3' UTR"]
initialDB_df = initialDB_coords_df.copy()
for col in sequence_cols:
    initialDB_df[col] = initialDB_df[col].apply(intervals_to_sequence)

initialDB_df.to_csv('../data/initialDB_sequences.csv', index=False)
initialDB_df.head()

feature,transcript_id,5' UTR,CDS,3' UTR
0,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,
1,ENST00000426406.4,,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA
2,ENST00000332831.5,,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA
3,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,
4,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...
